## 1. Setup Path dan Library


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

SEARCH_DIRS = [Path("/kaggle/working"), Path("/kaggle/input"), Path("/mnt/data")]
OUTPUT_DIR  = Path("/kaggle/working/outputs_scenario1_evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Outputs dari Notebook 01–03


In [2]:
def find_file(filename):
    for base in SEARCH_DIRS:
        if not base.exists(): continue
        matches = list(base.rglob(filename))
        if matches: return matches[0]
    return None

file_map = {
    "lda"       : find_file("lda_scenario1_results.csv"),
    "pubmedbert": find_file("bertopic_pubmedbert_scenario1_results.csv"),
    "specter2"  : find_file("bertopic_specter2_scenario1_results.csv"),
}

missing = [k for k, v in file_map.items() if v is None]
if missing:
    raise FileNotFoundError(
        f"Output belum ditemukan: {missing}\n"
        "Pastikan notebook 01, 02, dan 03 sudah dijalankan terlebih dahulu."
    )

lda_df        = pd.read_csv(file_map["lda"])
pubmedbert_df = pd.read_csv(file_map["pubmedbert"])
specter2_df   = pd.read_csv(file_map["specter2"])

print(f"LDA rows={len(lda_df)} | PubMedBERT rows={len(pubmedbert_df)} | SPECTER2 rows={len(specter2_df)}")

LDA rows=11 | PubMedBERT rows=1 | SPECTER2 rows=1


## 3. Full Combined Table


In [3]:
combined = pd.concat([lda_df, pubmedbert_df, specter2_df], ignore_index=True)
display(combined)

,experiment,embedding_model,model,K,k_type,n_topics_found,outlier_rate,cv,topic_uniqueness,hungarian_f1,harmonic_mean_cv_f1,best_min_cluster_size
0,LDA,-,LDA,5,k_search,5,0.0000,0.5141,0.8000,0.1395,NaN,NaN
1,LDA,-,LDA,10,k_search,10,0.0000,0.5809,0.8300,0.2794,NaN,NaN
2,LDA,-,LDA,15,k_search,15,0.0000,0.6331,0.8000,0.2860,NaN,NaN
3,LDA,-,LDA,20,k_search,20,0.0000,0.5948,0.7600,0.2423,NaN,NaN
4,LDA,-,LDA,25,k_search,25,0.0000,0.6017,0.7680,0.2762,NaN,NaN
5,LDA,-,LDA,30,k_search,30,0.0000,0.6377,0.8033,0.2656,NaN,NaN
6,LDA,-,LDA,40,k_search,40,0.0000,0.6084,0.7675,0.2289,NaN,NaN
7,LDA,-,LDA,50,k_search,50,0.0000,0.6289,0.7660,0.2382,NaN,NaN
8,LDA,-,LDA,60,k_search,60,0.0000,0.6277,0.7700,0.2565,NaN,NaN
9,LDA,-,LDA,75,k_search,75,0.0000,0.6014,0.7587,0.2214,NaN,NaN


## 4. Best Models per Method


In [4]:
# LDA: 2 baris — Cv dan F1 bisa menunjuk K berbeda
best_lda_cv = lda_df.sort_values("cv",           ascending=False).head(1).assign(selection="best_lda_by_cv")
best_lda_f1 = lda_df.sort_values("hungarian_f1", ascending=False).head(1).assign(selection="best_lda_by_f1")

scenario1_summary = pd.concat([
    best_lda_cv,
    best_lda_f1,
    pubmedbert_df.assign(selection="natural_hdbscan"),
    specter2_df.assign(selection="natural_hdbscan"),
], ignore_index=True)

display(scenario1_summary)

,experiment,embedding_model,model,K,k_type,n_topics_found,outlier_rate,cv,topic_uniqueness,hungarian_f1,selection,harmonic_mean_cv_f1,best_min_cluster_size
0,LDA,-,LDA,30,k_search,30,0.0000,0.6377,0.8033,0.2656,best_lda_by_cv,NaN,NaN
1,LDA,-,LDA,15,k_search,15,0.0000,0.6331,0.8000,0.2860,best_lda_by_f1,NaN,NaN
2,BERTopic_PubMedBERT,PubMedBERT,BERTopic_HDBSCAN_Natural,25,natural,25,0.2614,0.7215,0.7720,0.5377,natural_hdbscan,0.6162,100.0000
3,BERTopic_SPECTER2,SPECTER2,BERTopic_HDBSCAN_Natural,36,natural,36,0.2454,0.7398,0.7444,0.4909,natural_hdbscan,0.5902,50.0000


## 5. Ranking


In [5]:
COLS = ["experiment", "embedding_model", "model", "K", "cv", "topic_uniqueness", "hungarian_f1", "outlier_rate"]

rank_base = pd.concat([
    best_lda_cv,
    best_lda_f1,
    pubmedbert_df.assign(selection="natural_hdbscan"),
    specter2_df.assign(selection="natural_hdbscan"),
], ignore_index=True)

print("=== Ranking by Cv ===")
display(rank_base.sort_values("cv", ascending=False)[COLS])

print("\n=== Ranking by Hungarian F1 ===")
display(rank_base.sort_values("hungarian_f1", ascending=False)[COLS])

=== Ranking by Cv ===


,experiment,embedding_model,model,K,cv,topic_uniqueness,hungarian_f1,outlier_rate
3,BERTopic_SPECTER2,SPECTER2,BERTopic_HDBSCAN_Natural,36,0.7398,0.7444,0.4909,0.2454
2,BERTopic_PubMedBERT,PubMedBERT,BERTopic_HDBSCAN_Natural,25,0.7215,0.7720,0.5377,0.2614
0,LDA,-,LDA,30,0.6377,0.8033,0.2656,0.0000
1,LDA,-,LDA,15,0.6331,0.8000,0.2860,0.0000



=== Ranking by Hungarian F1 ===


,experiment,embedding_model,model,K,cv,topic_uniqueness,hungarian_f1,outlier_rate
2,BERTopic_PubMedBERT,PubMedBERT,BERTopic_HDBSCAN_Natural,25,0.7215,0.7720,0.5377,0.2614
3,BERTopic_SPECTER2,SPECTER2,BERTopic_HDBSCAN_Natural,36,0.7398,0.7444,0.4909,0.2454
1,LDA,-,LDA,15,0.6331,0.8000,0.2860,0.0000
0,LDA,-,LDA,30,0.6377,0.8033,0.2656,0.0000


## 6. Winner Selection → untuk Skenario 2


In [6]:
# Harmonic mean(Cv, F1) — keduanya harus bagus sekaligus
bertopic_all = pd.concat([pubmedbert_df, specter2_df], ignore_index=True)
bertopic_all["harmonic_mean_cv_f1"] = (
    2 * bertopic_all["cv"] * bertopic_all["hungarian_f1"] /
    (bertopic_all["cv"] + bertopic_all["hungarian_f1"])
).round(4)

winner = bertopic_all.sort_values("harmonic_mean_cv_f1", ascending=False).iloc[0]

print("=" * 50)
print(f"WINNER Skenario 1 (BERTopic):")
print(f"  Embedding  : {winner['embedding_model']}")
print(f"  Cv         : {winner['cv']}")
print(f"  F1         : {winner['hungarian_f1']}")
print(f"  HM(Cv,F1)  : {winner['harmonic_mean_cv_f1']}")
print(f"  Natural K  : {int(winner['K'])}")
print(f"  Best MCS   : {int(winner.get('best_min_cluster_size', 0))}")
print("=" * 50)

pd.DataFrame([winner]).to_csv(OUTPUT_DIR / "scenario1_winner_for_scenario2.csv", index=False)
print("Winner saved → akan dibaca oleh notebook Skenario 2.")

WINNER Skenario 1 (BERTopic):
  Embedding  : PubMedBERT
  Cv         : 0.7215
  F1         : 0.5377
  HM(Cv,F1)  : 0.6162
  Natural K  : 25
  Best MCS   : 100
Winner saved → akan dibaca oleh notebook Skenario 2.


## 7. Save All Outputs


In [7]:
combined.to_csv(          OUTPUT_DIR / "scenario1_all_results.csv",             index=False)
scenario1_summary.to_csv( OUTPUT_DIR / "scenario1_summary_best_models.csv",     index=False)
rank_base.sort_values("cv",           ascending=False).to_csv(OUTPUT_DIR / "scenario1_rank_by_cv.csv",           index=False)
rank_base.sort_values("hungarian_f1", ascending=False).to_csv(OUTPUT_DIR / "scenario1_rank_by_hungarian_f1.csv", index=False)

print(f"Saved to {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

Saved to /kaggle/working/outputs_scenario1_evaluation
  scenario1_all_results.csv
  scenario1_rank_by_cv.csv
  scenario1_rank_by_hungarian_f1.csv
  scenario1_summary_best_models.csv
  scenario1_winner_for_scenario2.csv
